# LifeMate (Zalo AI Challenge 2026 — Track 1) — Google Colab Public Test Notebook

Notebook này chạy pipeline **`Combined`** (`Qwen/Qwen3.5-4B` + ensemble 3 LoRA `hugebenevolence/lifemate-final` + ASR `asr-gipformer` + guardrails làm sạch tham số / dừng khi `sys.ask_missing_info` / gợi ý khi tool rỗng) để tạo và kiểm tra file **`submission.jsonl`** nộp lên **Public Test**.

- **Tổng tham số:** ~4.37B (3 LoRA) hoặc ~4.31B (1 LoRA), dưới giới hạn 6B của BTC.
- **Không cần finetune lại:** tải trực tiếp checkpoint `hugebenevolence/lifemate-final` và base `Qwen/Qwen3.5-4B` đã pin revision, tự động kiểm tra `SHA256SUMS`.
- **Hỗ trợ GPU Colab:**
  - **L4 (24 GB, SM 8.9) — Khuyến nghị:** chạy mặc định 3-LoRA ensemble, `--gpu-memory-utilization 0.85`, deadline 27s/mẫu.
  - **T4 (16 GB, SM 7.5):** tự động bật `--dtype float16 --enforce-eager --max-num-seqs 2 --gpu-memory-utilization 0.88` (tránh crash `bfloat16` và OOM). Mặc định đặt deadline 60s cho Public Test trên T4 để không bị cắt dở mẫu chậm; nếu muốn nhanh hơn ~3x, có thể đổi `USE_ENSEMBLE = False` ở Cell 2.

### Chuẩn bị trước khi chạy
1. Chọn **Runtime -> Change runtime type -> L4 GPU** (hoặc **T4 GPU**).
2. Đưa lên Google Drive (hoặc upload vào `/content`):
   - Thư mục code `Combined` (từ máy bạn).
   - Bộ dữ liệu `train+public_test` của BTC (tối thiểu phải có thư mục `public_test/` chứa `public_test.jsonl`, `public_test_tools_catalog.json`, `zaic_tools/`, `audio/`).


In [ ]:
# Cell 2: Cấu hình người dùng, kiểm tra GPU (qua nvidia-smi, chưa import torch), mount Drive & chuẩn bị workspace
import os
import shutil
import subprocess
import sys
from pathlib import Path

# === CẤU HÌNH CÓ THỂ CHỈNH ===
USE_ENSEMBLE = True          # True: vote 3 LoRA (lifemate:E1P,lifemate_e1:E1,lifemate_synth:E1); False: chỉ chạy 1 LoRA lifemate:E1P
MOUNT_DRIVE = True           # Tự mount Google Drive nếu đang chạy trên Colab
CODE_SOURCE = ""             # Để trống để tự tìm; hoặc điền path cụ thể VD: "/content/drive/MyDrive/Lifemate/Combined"
DATA_SOURCE = ""             # Để trống để tự tìm; hoặc điền path tới train+public_test hoặc public_test
WORK_DIR = Path("/content/lifemate_work" if Path("/content").exists() else "./lifemate_work").resolve()
COPY_DATA_TO_LOCAL = True    # Copy public_test từ Drive về ổ SSD cục bộ để đọc 634 file audio nhanh hơn
DEADLINE_SECONDS = None      # None = tự chọn (27.0s trên GPU >=20GB như L4; 60.0s trên T4 16GB cho Public Test)
# ==============================

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_DRIVE and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# 1. Kiểm tra GPU qua nvidia-smi (không import torch trước khi pip install vLLM)
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True,
)
gpu_line = smi.stdout.strip().splitlines()[0]
gpu_name, gpu_vram_mib_s, gpu_cc_s = [p.strip() for p in gpu_line.split(",")]
GPU_VRAM_MIB = int(float(gpu_vram_mib_s))
GPU_CC = tuple(int(x) for x in gpu_cc_s.split(".")[:2])
IS_T4_OR_LOW_CAP = (GPU_CC[0] < 8) or (GPU_VRAM_MIB < 20000)

if DEADLINE_SECONDS is None:
    DEADLINE_SECONDS = 60.0 if IS_T4_OR_LOW_CAP else 27.0

disk_root = Path("/content" if Path("/content").exists() else ".")
free_gb = shutil.disk_usage(disk_root).free / (1024 ** 3)
print(f"GPU: {gpu_name} | VRAM: {GPU_VRAM_MIB} MiB | Compute Cap: {GPU_CC[0]}.{GPU_CC[1]}")
print(f"Profile: {'T4 / Low-VRAM (<20GB or SM<8.0)' if IS_T4_OR_LOW_CAP else 'L4 / High-VRAM (>=20GB & SM>=8.0)'}")
print(f"USE_ENSEMBLE={USE_ENSEMBLE} | DEADLINE_SECONDS={DEADLINE_SECONDS}s | Disk free: {free_gb:.1f} GiB")
if free_gb < 25:
    print("CẢNH BÁO: Ổ đĩa còn < 25 GiB, có thể thiếu chỗ khi tải Qwen3.5-4B + audio.")

def find_code_dir(explicit: str) -> Path:
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    candidates += [
        Path.cwd(),
        Path("/content/Combined"),
        Path("/content/Lifemate/Combined"),
        Path("/content/drive/MyDrive/Lifemate/Combined"),
        Path("/content/drive/MyDrive/Combined"),
    ]
    for root in [Path("/content"), Path("/content/drive/MyDrive")]:
        if root.exists():
            for p in root.glob("*"):
                if p.is_dir():
                    candidates.append(p)
                    candidates.append(p / "Combined")
    for c in candidates:
        if (c / "src/lifemate/docker_predict.py").is_file() and (c / "configs/release.sh").is_file():
            return c.resolve()
    raise FileNotFoundError(
        "Không tìm thấy thư mục code Combined (chứa src/lifemate/docker_predict.py và configs/release.sh). "
        "Hãy upload Combined lên Google Drive hoặc điền CODE_SOURCE."
    )

def _walk_public_test(base_dir: Path, max_depth: int = 6) -> list[Path]:
    skip_dirs = {
        "__MACOSX", ".git", ".venv", ".venv-asr", "audio",
        "public_test_environment", "train", "train_environment",
        "models", "outputs", "src", "docs", "synth", "analysis",
        "scripts", "tests", "docker", "lifemate_work", "sample_data",
    }
    found: list[Path] = []
    if not base_dir.is_dir():
        return found
    base_parts = len(base_dir.parts)
    for cur_root, dirs, files in os.walk(base_dir):
        cur_path = Path(cur_root)
        if len(cur_path.parts) - base_parts > max_depth:
            dirs[:] = []
            continue
        dirs[:] = sorted(d for d in dirs if d not in skip_dirs and not d.startswith("."))
        if "public_test.jsonl" in files and "public_test_tools_catalog.json" in files:
            found.append(cur_path)
            dirs[:] = []
    return found

def find_public_test_dir(explicit: str, code_dir: Path) -> Path:
    candidates = []
    if explicit:
        p = Path(explicit)
        candidates += [p, p / "public_test", *_walk_public_test(p)]
    candidates += [
        code_dir / "data/train+public_test/public_test",
        code_dir.parent / "train+public_test/public_test",
        code_dir.parent / "data/train+public_test/public_test",
        code_dir.parent / "public_test",
        Path("/content/train+public_test/public_test"),
        Path("/content/data/train+public_test/public_test"),
        Path("/content/public_test"),
        Path("/content/drive/MyDrive/Lifemate/train+public_test/public_test"),
        Path("/content/drive/MyDrive/Lifemate/data/train+public_test/public_test"),
        Path("/content/drive/MyDrive/Colab_Upload/train+public_test/public_test"),
        Path("/content/drive/MyDrive/train+public_test/public_test"),
        Path("/content/drive/MyDrive/public_test"),
    ]
    for c in candidates:
        if (c / "public_test.jsonl").is_file() and (c / "public_test_tools_catalog.json").is_file():
            return c.resolve()
    for base in (code_dir, code_dir.parent):
        for c in _walk_public_test(base):
            return c.resolve()
    for root in (Path("/content"), Path("/content/drive/MyDrive")):
        if not root.exists():
            continue
        for c in _walk_public_test(root, max_depth=3):
            return c.resolve()
        for child in sorted(root.iterdir()):
            if child.is_dir() and any(k in child.name.lower() for k in ("lifemate", "colab", "zai", "train", "public")):
                for c in _walk_public_test(child, max_depth=6):
                    return c.resolve()
    raise FileNotFoundError(
        "Không tìm thấy thư mục public_test (cần public_test.jsonl, public_test_tools_catalog.json, zaic_tools/, audio/). "
        "Hãy đưa bộ dữ liệu train+public_test từ BTC lên Drive hoặc điền DATA_SOURCE."
    )

CODE_DIR = find_code_dir(CODE_SOURCE)
SRC_PUBLIC_TEST_DIR = find_public_test_dir(DATA_SOURCE, CODE_DIR)
print(f"Code source: {CODE_DIR}")
print(f"Public test source: {SRC_PUBLIC_TEST_DIR}")

# Copy code sang WORK_DIR (bỏ qua data/models/outputs cũ nếu có)
WORK_DIR.mkdir(parents=True, exist_ok=True)
if CODE_DIR != WORK_DIR:
    for item_name in ["src", "configs", "scripts", "synth", "tests", "pyproject.toml", "README.md"]:
        src_p = CODE_DIR / item_name
        dst_p = WORK_DIR / item_name
        if not src_p.exists():
            continue
        if src_p.is_dir():
            shutil.copytree(src_p, dst_p, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
        else:
            shutil.copy2(src_p, dst_p)

# Copy public_test từ Drive sang SSD local (đa luồng, bỏ qua copystat FUSE để nhanh gấp ~10x)
if COPY_DATA_TO_LOCAL and str(SRC_PUBLIC_TEST_DIR).startswith("/content/drive"):
    LOCAL_PUBLIC_TEST_DIR = WORK_DIR / "data_local/public_test"
    marker = LOCAL_PUBLIC_TEST_DIR / ".copy_complete"
    if not (marker.is_file() and (LOCAL_PUBLIC_TEST_DIR / "public_test.jsonl").is_file()):
        from concurrent.futures import ThreadPoolExecutor
        import time as _time

        t_cp = _time.perf_counter()
        jobs = []
        for cur_root, dirs, files in os.walk(SRC_PUBLIC_TEST_DIR):
            dirs[:] = [d for d in dirs if d != "__MACOSX"]
            rel = Path(cur_root).relative_to(SRC_PUBLIC_TEST_DIR)
            (LOCAL_PUBLIC_TEST_DIR / rel).mkdir(parents=True, exist_ok=True)
            for fn in files:
                if fn != ".DS_Store" and not fn.startswith("._"):
                    jobs.append((Path(cur_root) / fn, LOCAL_PUBLIC_TEST_DIR / rel / fn))
        print(f"Đang copy {len(jobs)} files từ Drive sang {LOCAL_PUBLIC_TEST_DIR} (16 luồng)...")
        with ThreadPoolExecutor(max_workers=16) as ex:
            list(ex.map(lambda p: shutil.copyfile(p[0], p[1]), jobs))
        marker.write_text("ok\n", encoding="utf-8")
        print(f"Copy public_test xong trong {_time.perf_counter() - t_cp:.1f}s")
else:
    LOCAL_PUBLIC_TEST_DIR = SRC_PUBLIC_TEST_DIR

# Stage thư mục chỉ chứa 1 split public_test (đúng chuẩn lifemate.docker_predict)
STAGE_DATA_DIR = WORK_DIR / "stage_public_test"
STAGE_DATA_DIR.mkdir(parents=True, exist_ok=True)
stage_split_link = STAGE_DATA_DIR / "public_test"
if stage_split_link.is_symlink() or stage_split_link.exists():
    if stage_split_link.is_symlink():
        stage_split_link.unlink()
    else:
        shutil.rmtree(stage_split_link)
stage_split_link.symlink_to(LOCAL_PUBLIC_TEST_DIR.resolve())

os.environ["TZ"] = "Asia/Ho_Chi_Minh"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["VLLM_NO_USAGE_STATS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if str(WORK_DIR / "src") not in sys.path:
    sys.path.insert(0, str(WORK_DIR / "src"))
os.chdir(WORK_DIR)
print(f"WORK_DIR ready: {WORK_DIR}")
print(f"STAGE_DATA_DIR ready: {STAGE_DATA_DIR} -> {LOCAL_PUBLIC_TEST_DIR}")


In [ ]:
# Cell 3: Cài đặt thư viện đúng phiên bản từ pyproject.toml (vllm==0.32.0, sherpa-onnx==2.0.1, soundfile==0.14.0)
import importlib.metadata
import subprocess
import sys

print("Python:", sys.version)
assert (3, 11) <= sys.version_info[:2] < (3, 14), f"Cần Python >=3.11,<3.14, hiện tại: {sys.version}"

def _run_pip(args):
    proc = subprocess.Popen(
        [sys.executable, "-m", "pip", *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, proc.args)

# Lưu ý: KHÔNG pin huggingface_hub==0.36.0 chung môi trường vì vllm==0.32.0 yêu cầu huggingface_hub>=1.31.0
REQUIRED = {
    "vllm": "0.32.0",
    "sherpa-onnx": "2.0.1",
    "soundfile": "0.14.0",
}

missing_or_mismatch = []
for pkg, ver in REQUIRED.items():
    try:
        cur = importlib.metadata.version(pkg)
        if cur != ver:
            missing_or_mismatch.append(f"{pkg}=={ver}")
    except importlib.metadata.PackageNotFoundError:
        missing_or_mismatch.append(f"{pkg}=={ver}")

if missing_or_mismatch:
    print("Đang cài đặt:", missing_or_mismatch)
    _run_pip(["install", "--disable-pip-version-check", *missing_or_mismatch])
else:
    print("Các thư viện chính đã đúng phiên bản.")

_run_pip(["install", "--disable-pip-version-check", "-q", "--no-deps", "--no-build-isolation", "-e", str(WORK_DIR)])

try:
    import torch
    import vllm
    import sherpa_onnx
    import soundfile
    import huggingface_hub
    import lifemate.agent
    import lifemate.ensemble
    import lifemate.docker_predict
    import lifemate.submission
except Exception as exc:
    raise RuntimeError(
        f"Lỗi import sau khi cài thư viện ({exc}). "
        "Nếu Colab vừa nâng cấp torch/numpy, hãy bấm Runtime -> Restart session rồi chạy lại từ Cell 2."
    ) from exc

print(
    f"OK | torch={torch.__version__} | cuda={torch.cuda.is_available()} "
    f"| vllm={vllm.__version__} | sherpa_onnx={sherpa_onnx.__version__} "
    f"| soundfile={soundfile.__version__} | huggingface_hub={huggingface_hub.__version__}"
)
assert torch.cuda.is_available(), "PyTorch không thấy CUDA GPU! Hãy bật GPU Runtime hoặc Runtime -> Restart session."


In [ ]:
# Cell 4: Kiểm tra bộ dữ liệu public_test & zaic_tools sandbox
from lifemate.data import audio_path, catalog, installed_tools, items
from lifemate.toolenv import new_env

required_files = [
    STAGE_DATA_DIR / "public_test/public_test.jsonl",
    STAGE_DATA_DIR / "public_test/public_test_tools_catalog.json",
    STAGE_DATA_DIR / "public_test/audio",
    STAGE_DATA_DIR / "public_test/zaic_tools",
]
for p in required_files:
    assert p.exists(), f"Thiếu file/thư mục bắt buộc trong kit: {p}"

samples_preview = list(items(STAGE_DATA_DIR, "public_test"))
specs_preview = catalog(STAGE_DATA_DIR, "public_test")
first_item = samples_preview[0]
first_wav = audio_path(STAGE_DATA_DIR, "public_test", first_item["audio"])
test_env = new_env(STAGE_DATA_DIR, "public_test", first_item["id"])

print(f"Số mẫu public_test: {len(samples_preview)} | Số tools trong catalog: {len(specs_preview)}")
print(f"Mẫu đầu tiên: id={first_item['id']} | wav={first_wav.name} | installed_tools={len(installed_tools(first_item, specs_preview))}")
print(f"zaic_tools ToolEnv OK | initial_state_hash={test_env.get_state_hash()}")


In [ ]:
# Cell 5: Tải base model Qwen/Qwen3.5-4B (pin revision) & release checkpoint hugebenevolence/lifemate-final + check SHA256SUMS
import hashlib
import os
from pathlib import Path
from huggingface_hub import snapshot_download

os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)

BASE_REPO = "Qwen/Qwen3.5-4B"
BASE_REV = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
RELEASE_REPO = "hugebenevolence/lifemate-final"

BASE_MODEL_DIR = WORK_DIR / "models/qwen3.5-4b"
RELEASE_DIR = WORK_DIR / "outputs/release/lifemate-final"
BASE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
RELEASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Đang tải/kiểm tra {BASE_REPO} @ {BASE_REV[:8]} -> {BASE_MODEL_DIR}")
snapshot_download(repo_id=BASE_REPO, revision=BASE_REV, local_dir=str(BASE_MODEL_DIR))

print(f"Đang tải/kiểm tra {RELEASE_REPO} -> {RELEASE_DIR}")
snapshot_download(repo_id=RELEASE_REPO, local_dir=str(RELEASE_DIR))

# Kiểm tra SHA256SUMS của release checkpoint
sums_file = RELEASE_DIR / "SHA256SUMS"
assert sums_file.is_file(), f"Không thấy {sums_file}"
verified_count = 0
for line in sums_file.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line:
        continue
    expected_hex, rel_path = line.split(None, 1)
    rel_path = rel_path.lstrip("*")
    target = RELEASE_DIR / rel_path
    assert target.is_file(), f"Thiếu file trong checkpoint: {target}"
    h = hashlib.sha256()
    with target.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    actual_hex = h.hexdigest()
    if actual_hex != expected_hex:
        raise RuntimeError(f"Sai SHA256 cho {rel_path}: expected {expected_hex}, got {actual_hex}")
    verified_count += 1

ASR_DIR = RELEASE_DIR / "asr-gipformer"
for req_asr in ("encoder.onnx", "decoder.onnx", "joiner.onnx", "tokens.txt"):
    assert (ASR_DIR / req_asr).is_file(), f"Thiếu file ASR {req_asr} trong {ASR_DIR}"

# Tùy chọn: trỏ tới LoRA vừa tự train từ colab_train_g4.ipynb (đặt None để dùng bản release gốc)
CUSTOM_LORA_P1_DIR = None  # Ví dụ: DRIVE_ROOT / "trained_outputs/lifemate-v2/llm-lora-p1"
if CUSTOM_LORA_P1_DIR is not None:
    import shutil
    custom_src = Path(CUSTOM_LORA_P1_DIR)
    assert (custom_src / "adapter_model.safetensors").is_file(), f"Không thấy adapter mới tại {custom_src}"
    dst_p1 = RELEASE_DIR / "llm-lora-p1"
    if dst_p1.exists():
        shutil.rmtree(dst_p1)
    shutil.copytree(custom_src, dst_p1)
    print(f"Đã thay llm-lora-p1 bằng adapter mới: {custom_src}")

REQUIRED_LORAS = ["llm-lora-p1"] + (["llm-lora-e1", "llm-lora-synth"] if USE_ENSEMBLE else [])
for dirname in REQUIRED_LORAS:
    assert (RELEASE_DIR / dirname / "adapter_config.json").is_file(), f"Thiếu LoRA adapter: {RELEASE_DIR / dirname}"

print(f"SHA256SUMS OK ({verified_count} files verified).")
print(f"Model setup ready | USE_ENSEMBLE={USE_ENSEMBLE} | Adapters={REQUIRED_LORAS} (~{'4.37B' if USE_ENSEMBLE else '4.31B'} total params < 6B limit)")


In [ ]:
# Cell 6: Thiết lập biến môi trường LIFEMATE_* & khởi chạy local vLLM server
import json
import os
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

# Dừng server cũ nếu chạy lại Cell 6
old_server = globals().get("server")
if old_server is not None and old_server.poll() is None:
    print("Đang dừng vLLM server cũ...")
    old_server.terminate()
    try:
        old_server.wait(timeout=15)
    except subprocess.TimeoutExpired:
        old_server.kill()

# Chọn port trống bắt đầu từ 8000
VLLM_PORT = 8000
while True:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", VLLM_PORT)) != 0:
            break
    VLLM_PORT += 1
    assert VLLM_PORT < 8020, "Không tìm được port trống từ 8000-8019"

# Biến LIFEMATE_* khớp configs/release.sh
ENSEMBLE_STR = "lifemate:E1P,lifemate_e1:E1,lifemate_synth:E1" if USE_ENSEMBLE else ""
lora_modules = [f"lifemate={RELEASE_DIR / 'llm-lora-p1'}"]
if USE_ENSEMBLE:
    lora_modules += [
        f"lifemate_e1={RELEASE_DIR / 'llm-lora-e1'}",
        f"lifemate_synth={RELEASE_DIR / 'llm-lora-synth'}",
    ]

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["LIFEMATE_SERVED_MODEL"] = "lifemate"
os.environ["LIFEMATE_ASR_DIR"] = str(RELEASE_DIR / "asr-gipformer")
os.environ["LIFEMATE_ASR_THREADS"] = str(min(8, os.cpu_count() or 4))
os.environ["LIFEMATE_DEADLINE"] = str(DEADLINE_SECONDS)
os.environ["LIFEMATE_ORCHESTRATION"] = "E1P"
os.environ["LIFEMATE_RETRY_CAP"] = "2"
os.environ["LIFEMATE_MAX_TURNS"] = "10"
os.environ["LIFEMATE_ENSEMBLE"] = ENSEMBLE_STR
os.environ["LIFEMATE_MERGE_SYSTEM"] = "1"
os.environ["LIFEMATE_LLM_EXTRA"] = '{"chat_template_kwargs": {"enable_thinking": false}}'

# Cờ vLLM tự điều chỉnh theo L4 vs T4
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(BASE_MODEL_DIR),
    "--served-model-name", "qwen35_base",
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--language-model-only",
    "--max-model-len", "16384",
    "--seed", "42",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--reasoning-parser", "qwen3",
    "--enable-lora",
    "--max-lora-rank", "32",
    "--max-loras", str(len(lora_modules)),
    "--lora-modules", *lora_modules,
]

if IS_T4_OR_LOW_CAP:
    vllm_cmd += [
        "--dtype", "float16",
        "--enforce-eager",
        "--max-num-seqs", "2",
        "--gpu-memory-utilization", "0.88",
    ]
else:
    vllm_cmd += ["--gpu-memory-utilization", "0.85"]

VLLM_LOG = WORK_DIR / "outputs/vllm.log"
VLLM_LOG.parent.mkdir(parents=True, exist_ok=True)
print("Khởi chạy vLLM:", " ".join(vllm_cmd))
log_handle = VLLM_LOG.open("w", encoding="utf-8")
server = subprocess.Popen(vllm_cmd, stdout=log_handle, stderr=subprocess.STDOUT)

LLM_CHAT_URL = f"http://127.0.0.1:{VLLM_PORT}/v1/chat/completions"
MODELS_URL = f"http://127.0.0.1:{VLLM_PORT}/v1/models"

t0 = time.perf_counter()
timeout_s = 420
served_ids = []
while time.perf_counter() - t0 < timeout_s:
    if server.poll() is not None:
        tail = "\n".join(VLLM_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-80:])
        raise RuntimeError(f"vLLM server thoát sớm (code {server.returncode}). Log cuối:\n{tail}")
    try:
        with urllib.request.urlopen(MODELS_URL, timeout=2) as resp:
            data = json.loads(resp.read().decode("utf-8"))
            served_ids = [m["id"] for m in data.get("data", [])]
            break
    except Exception:
        time.sleep(3)
else:
    server.terminate()
    tail = "\n".join(VLLM_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-80:])
    raise TimeoutError(f"vLLM server chưa sẵn sàng sau {timeout_s}s. Log cuối:\n{tail}")

expected_ids = {"lifemate", "lifemate_e1", "lifemate_synth"} if USE_ENSEMBLE else {"lifemate"}
assert expected_ids.issubset(set(served_ids)), f"Thiếu model trên vLLM: cần {expected_ids}, có {served_ids}"
print(f"vLLM sẵn sàng sau {time.perf_counter() - t0:.1f}s | URL={LLM_CHAT_URL} | Models={served_ids}")


In [ ]:
# Cell 7: Smoke test nhanh (mặc định 10 mẫu) & kiểm tra tốc độ / validate trước khi chạy full 634 mẫu
import json
import os
import random
import time
import numpy as np
import torch
from pathlib import Path
from lifemate.docker_predict import Predictor
from lifemate.submission import validate_file

SMOKE_LIMIT = 10  # Đổi thành 0 nếu muốn bỏ qua smoke test và chạy thẳng Cell 8

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = "0"
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
assert server.poll() is None, "vLLM server đang không chạy — hãy chạy lại Cell 6!"

pred = Predictor(
    STAGE_DATA_DIR,
    LLM_CHAT_URL,
    os.environ["LIFEMATE_SERVED_MODEL"],
    Path(os.environ["LIFEMATE_ASR_DIR"]),
    float(os.environ["LIFEMATE_DEADLINE"]),
)
all_samples = pred.samples()
print(f"Predictor sẵn sàng | split={pred.split} | tổng={len(all_samples)} mẫu | orchestration={pred.orchestration} | retry_cap={pred.retry_cap} | ensemble={bool(os.environ.get('LIFEMATE_ENSEMBLE'))}")

if SMOKE_LIMIT > 0:
    smoke_dir = WORK_DIR / "outputs/colab_smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    smoke_time_path = smoke_dir / "submission_with_time.jsonl"
    smoke_clean_path = smoke_dir / "submission_clean.jsonl"
    smoke_rows = []
    with smoke_time_path.open("w", encoding="utf-8") as f_time, smoke_clean_path.open("w", encoding="utf-8") as f_clean:
        for idx, item in enumerate(all_samples[:SMOKE_LIMIT], 1):
            row = pred.predict(item)
            smoke_rows.append(row)
            f_time.write(json.dumps(row, ensure_ascii=False) + "\n")
            f_time.flush()
            clean_row = {"id": row["id"], "tool_calls": row["tool_calls"], "state_hash": row["state_hash"]}
            f_clean.write(json.dumps(clean_row, ensure_ascii=False) + "\n")
            f_clean.flush()
            print(f"  [{idx}/{SMOKE_LIMIT}] id={row['id']} | time={row['time']} ms | tool_calls={len(row['tool_calls'])}")

    check = validate_file(STAGE_DATA_DIR, "public_test", smoke_clean_path, allow_partial=True)
    times = [r["time"] for r in smoke_rows]
    est_min = (sum(times) / len(times)) * len(all_samples) / 60000.0
    print(f"Smoke test VALIDATE OK: {check}")
    print(f"Tốc độ smoke test: mean={sum(times)/len(times):.0f} ms | max={max(times)} ms | Ước tính chạy full {len(all_samples)} mẫu: ~{est_min:.1f} phút")
else:
    print("Bỏ qua smoke test (SMOKE_LIMIT=0).")


In [ ]:
# Cell 8: Chạy toàn bộ public_test (634 mẫu), lưu checkpoint liên tục, validate bằng ToolEnv & tắt server
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
import torch
from lifemate.docker_predict import Predictor
from lifemate.submission import validate_file

RESUME_EXISTING = True  # True: tự nối tiếp nếu đã có submission_with_time.jsonl; False: chạy lại sạch từ đầu

assert server.poll() is None, "vLLM server đang không chạy — hãy chạy lại Cell 6!"
random.seed(42)
os.environ["PYTHONHASHSEED"] = "0"
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Khởi tạo Predictor sạch cho lượt chạy chính
pred = Predictor(
    STAGE_DATA_DIR,
    LLM_CHAT_URL,
    os.environ["LIFEMATE_SERVED_MODEL"],
    Path(os.environ["LIFEMATE_ASR_DIR"]),
    float(os.environ["LIFEMATE_DEADLINE"]),
)
samples = pred.samples()
sample_ids = {s["id"] for s in samples}

out_dir = WORK_DIR / "outputs/colab_public"
out_dir.mkdir(parents=True, exist_ok=True)
time_path = out_dir / "submission_with_time.jsonl"
final_sub_path = out_dir / "submission.jsonl"

drive_backup_dir = None
if Path("/content/drive/MyDrive").exists():
    drive_backup_dir = Path("/content/drive/MyDrive/Lifemate/colab_outputs")
    drive_backup_dir.mkdir(parents=True, exist_ok=True)

done_by_id = {}
if RESUME_EXISTING and time_path.is_file():
    for raw_line in time_path.read_text(encoding="utf-8").splitlines():
        if not raw_line.strip():
            continue
        try:
            r = json.loads(raw_line)
            if r.get("id") in sample_ids and {"id", "tool_calls", "state_hash", "time"}.issubset(r):
                done_by_id[r["id"]] = r
        except json.JSONDecodeError:
            continue
    print(f"Resume: đã có {len(done_by_id)}/{len(samples)} mẫu hợp lệ trong {time_path}")
elif time_path.exists():
    time_path.unlink()

# Ghi lại bản sạch các mẫu đã resume trước khi tiếp tục
with time_path.open("w", encoding="utf-8") as fout:
    for item in samples:
        if item["id"] in done_by_id:
            fout.write(json.dumps(done_by_id[item["id"]], ensure_ascii=False) + "\n")
    fout.flush()

    remaining = [s for s in samples if s["id"] not in done_by_id]
    print(f"Cần chạy thêm {len(remaining)}/{len(samples)} mẫu...")
    t_start = time.perf_counter()
    new_times = []

    for idx, item in enumerate(remaining, 1):
        row = pred.predict(item)
        done_by_id[row["id"]] = row
        fout.write(json.dumps(row, ensure_ascii=False) + "\n")
        fout.flush()
        new_times.append(row["time"])

        if idx % 10 == 0 or idx == len(remaining):
            total_done = len(done_by_id)
            avg_ms = sum(new_times) / len(new_times)
            eta_min = avg_ms * (len(remaining) - idx) / 60000.0
            print(
                f"  [{total_done}/{len(samples)}] last={row['id']} "
                f"| time={row['time']} ms | avg={avg_ms:.0f} ms | ETA={eta_min:.1f}m"
            )
        if drive_backup_dir is not None and (idx % 25 == 0 or idx == len(remaining)):
            shutil.copy2(time_path, drive_backup_dir / "submission_with_time.jsonl")

# Xuất file submission.jsonl chuẩn Public Test (đúng thứ tự mẫu, đúng 3 key: id, tool_calls, state_hash)
tmp_sub = out_dir / "submission.jsonl.tmp"
with tmp_sub.open("w", encoding="utf-8") as fsub:
    for item in samples:
        r = done_by_id[item["id"]]
        clean_row = {
            "id": r["id"],
            "tool_calls": r["tool_calls"],
            "state_hash": r["state_hash"],
        }
        fsub.write(json.dumps(clean_row, ensure_ascii=False) + "\n")

validation_report = validate_file(STAGE_DATA_DIR, "public_test", tmp_sub, allow_partial=False)
tmp_sub.rename(final_sub_path)

all_times = [done_by_id[s["id"]]["time"] for s in samples]
over_27s = sum(t > 27000 for t in all_times)
over_30s = sum(t > 30000 for t in all_times)
print("\n=== KẾT QUẢ PUBLIC TEST ===")
print("Validate (lifemate.submission.validate_file):", validation_report)
print(
    f"Tổng mẫu: {len(all_times)} | Mean: {sum(all_times)/len(all_times):.0f} ms "
    f"| Max: {max(all_times)} ms | >27s: {over_27s} | >30s: {over_30s}"
)
print(f"File nộp Public Test: {final_sub_path}")

# Kiểm tra thêm CLI của zaic_tools nếu kit có hỗ trợ --validate
env_check = {**os.environ, "PYTHONPATH": str((STAGE_DATA_DIR / "public_test").resolve())}
help_proc = subprocess.run([sys.executable, "-m", "zaic_tools", "--help"], env=env_check, capture_output=True, text=True)
if help_proc.returncode == 0 and "--validate" in help_proc.stdout:
    cmd = [
        sys.executable, "-m", "zaic_tools",
        "--split", "public_test",
        "--root", str(STAGE_DATA_DIR),
        "--validate", str(final_sub_path),
    ]
    z_proc = subprocess.run(cmd, env=env_check, capture_output=True, text=True)
    print("zaic_tools --validate exit:", z_proc.returncode)
    if z_proc.stdout:
        print(z_proc.stdout.strip())
    if z_proc.stderr:
        print(z_proc.stderr.strip())

# Dọn dẹp vLLM server sau khi hoàn tất
if server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server.kill()
print("Đã tắt vLLM server.")


In [ ]:
# Cell 9: Lưu file submission.jsonl lên Google Drive (nếu có) & tải về máy tính
import json
import shutil
import sys
from pathlib import Path

assert final_sub_path.is_file(), f"Chưa thấy {final_sub_path}"
lines = [l for l in final_sub_path.read_text(encoding="utf-8").splitlines() if l.strip()]
sample_row = json.loads(lines[0])
assert set(sample_row.keys()) == {"id", "tool_calls", "state_hash"}, f"Sai schema keys: {sample_row.keys()}"

size_kb = final_sub_path.stat().st_size / 1024.0
print(f"File hợp lệ: {final_sub_path} | {len(lines)} dòng | {size_kb:.1f} KiB | keys={sorted(sample_row.keys())}")

if Path("/content/drive/MyDrive").exists():
    drive_out = Path("/content/drive/MyDrive/Lifemate/colab_outputs")
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(final_sub_path, drive_out / "submission.jsonl")
    if time_path.is_file():
        shutil.copy2(time_path, drive_out / "submission_with_time.jsonl")
    print(f"Đã backup lên Google Drive: {drive_out / 'submission.jsonl'}")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(final_sub_path))
    print("Đã kích hoạt tải file submission.jsonl về trình duyệt.")
else:
    print(f"Không ở trong Google Colab UI; hãy lấy file trực tiếp tại: {final_sub_path}")


## Hướng dẫn nộp bài lên Zalo AI Challenge

1. **Nộp Public Test:**
   - Truy cập trang nộp bài Track 1 (**LifeMate**) trên cổng Zalo AI Challenge.
   - Tải lên đúng file **`submission.jsonl`** vừa download ở Cell 9 (mỗi dòng chỉ gồm 3 trường `id`, `tool_calls`, `state_hash`, đã pass `lifemate.submission.validate_file`).
   - *Lưu ý:* Không nộp file `submission_with_time.jsonl` (file đó chỉ giữ trường `time` dùng để kiểm tra độ trễ).

2. **Chuẩn bị nộp Private Test (nếu cần ở vòng sau):**
   - Private Test chấm bằng **Docker image** (`--network none`, GPU >= 24 GB, <= 30s/mẫu).
   - Toàn bộ code `Combined/` đã có sẵn [docker/Dockerfile](docker/Dockerfile), [docker/build.sh](docker/build.sh) và [configs/release.sh](configs/release.sh) để build image từ cùng cấu hình `Combined`.
